# ND2 Unified Discovery Notebook: Fase 6 - Generalización y Recurrencia de Legendre

Objetivo Final: Integrar la ley radial descubierta ($1/r^{l+1}$) para aislar y resolver el Polinomio de Legendre general $P(l, \theta)$.
Target: V * r^(l+1)
Variables: l_order, theta

In [ ]:
# 0. Sincronización Robusta (Hard Reset)
import os
is_colab = os.path.exists('/content')
base_path = '/content' if is_colab else '/kaggle/working'
target_dir = os.path.join(base_path, 'ND2')
if not os.path.exists(target_dir): os.system(f"git clone https://github.com/manramirezpi/ND2.git {target_dir}")
os.chdir(target_dir)
os.system("git fetch origin")
os.system("git reset --hard origin/main")
print(f"\n[OK] Entorno Fase 6 listo en: {os.getcwd()}")

In [ ]:
# 1. Setup
!pip install gplearn setproctitle geopandas
!mkdir -p weights Multipolos/data/nd2_format Multipolos/results
if not os.path.exists('weights/checkpoint.pth'):
    !wget -O weights/checkpoint.pth https://github.com/yuzhTHU/ND2/releases/download/checkpoint.pth/checkpoint.pth

In [ ]:
# 2. GENERAR DATASET MULTI-ORDEN CON COMPENSACIÓN RADIAL
import numpy as np, json, os
from scipy.special import legendre

def generate_multi_order_reduced_data(num_samples_per_l=800):
    all_l = [1, 2, 3, 4, 5]
    
    data_node = {"l_order": [], "theta": []}
    targets = []
    
    for l in all_l:
        r = np.random.uniform(1.0, 5.0, num_samples_per_l)
        theta = np.random.uniform(0, np.pi, num_samples_per_l)
        
        x = np.cos(theta)
        Pl = legendre(l)(x)
        V = Pl / (r**(l+1)) # La ley física real
        
        # COMPENSACIÓN RADIAL: El target es exactamente Pl
        # Esto es lo que el usuario llama 'generalizar certeramente el componente radial'
        V_reduced = V * (r**(l+1))
        
        data_node["l_order"].extend([[0.0, float(l)] for _ in range(num_samples_per_l)])
        data_node["theta"].extend([[0.0, float(t)] for t in theta])
        targets.extend([[0.0, float(vr)] for vr in V_reduced])
    
    dataset = {"V": 2, "E": 1, "A": [[0,1],[0,0]], "G": [[0,1]],
               "l_order": data_node["l_order"],
               "theta": data_node["theta"],
               "target": targets}
    
    os.makedirs("Multipolos/data/nd2_format", exist_ok=True)
    json.dump(dataset, open("Multipolos/data/nd2_format/MULTI_L_REDUCED_nd2.json", "w"))
    print(f"Dataset Multi-l (l={all_l}) con compensación radial generado.")

generate_multi_order_reduced_data()

In [ ]:
# 3. BÚSQUEDA DE LA FUNCIÓN GENERAL P(l, theta)
!python3 search.py \
    --data Multipolos/data/nd2_format/MULTI_L_REDUCED_nd2.json \
    --vars l_order theta \
    --target_var target \
    --episodes 5000 \
    --beam_size 15 \
    --device cuda

In [ ]:
# 4. AUTO-PUSH
def get_token_reloaded():
    try: from kaggle_secrets import UserSecretsClient; return UserSecretsClient().get_secret("GITHUB_TOKEN")
    except: pass
    try: from google.colab import userdata; return userdata.get('GITHUB_TOKEN')
    except: pass
    return None
token = get_token_reloaded()
if token:
    os.system("git config --global user.email 'manuel@researcher.phys'")
    os.system("git config --global user.name 'ND2-Auto-Agent'")
    os.system(f"git remote set-url origin https://{token}@github.com/manramirezpi/ND2.git")
    !git add Multipolos/results/ Multipolos/data/nd2_format/MULTI_L_REDUCED_nd2.json
    !git commit -m "Fase 6: Búsqueda de Generalización de Legendre (Target Reducido)"
    !git push origin main
else: print("\n[!] GITHUB_TOKEN no encontrado.")